# Topic 07 — DateTime & Time Series

> **Investigation milestone:** The big break. Aurora's `order_date` is stored in
> *mixed formats*; once parsed correctly you can finally plot revenue over time —
> and the suspicious **dip** the finance team worried about becomes visible. Was
> it real, or a parsing artifact?

**Time split: 20% reading · 80% in `practice.ipynb`.**

---

## Parsing messy dates

```python
orders["order_date"] = pd.to_datetime(orders["order_date"],
                                       format="mixed", dayfirst=True,
                                       errors="coerce")
```
- `format="mixed"` handles rows with different formats (Aurora has 4).
- `dayfirst=True` reads `03/04/2023` as 3 April (European), not 4 March.
- `errors="coerce"` turns unparseable junk into `NaT` (missing datetime) instead
  of crashing — then you *count* the NaT to measure data quality.

> **WHY parse, not eyeball:** as text, `"12/2023"` sorts before `"2/2024"`
> alphabetically. A wrong dip in the chart is often just a wrong sort.

## The `.dt` accessor — pull parts out

```python
orders["year"]  = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.to_period("M")   # 2023-11
orders["dow"]   = orders["order_date"].dt.day_name()
orders["is_weekend"] = orders["order_date"].dt.weekday >= 5
```

## `resample` — groupby for time

Set a datetime index, then resample to any frequency:
```python
ts = order_rev.set_index("order_date")
monthly = ts["line_revenue"].resample("M").sum()   # ME in newer pandas
weekly  = ts["line_revenue"].resample("W").sum()
```
`resample` = "group by time bucket". `D`, `W`, `M`, `Q`, `Y` (and `ME`/`MS`).

## Rolling & shift — trends and growth

```python
monthly.rolling(3).mean()         # 3-month moving average (smooths noise)
monthly.pct_change()              # month-over-month growth
monthly.shift(1)                  # previous month's value (for comparisons)
monthly - monthly.shift(12)       # year-over-year change
```

## Gaps in time series

Aurora's `marketing_spend` is **missing a few days**. Reindex to a full date
range and decide: `fillna(0)`? interpolate? leave NaT? (Interview Q24).
```python
full = pd.date_range(start, end, freq="D")
daily = daily.reindex(full)
```

## NumPy connection 🔢

Datetimes are stored as `datetime64[ns]` — a NumPy dtype (int nanoseconds under
the hood). Differences give `timedelta64`. `rolling().mean()` is a vectorized
windowed reduction; `shift` is an array roll. Your NumPy intuition transfers.

## Visual learning 📊

The signature chart: `monthly.plot()` with a `rolling(3).mean()` overlay. This
is *the* chart that reveals (or debunks) the revenue dip. Add it to the capstone.

---

## 🔎 Interview Lens (answer in writing)
- **Q22:** Why is `parse_dates`/parsing at load better than converting after? When is the reverse true?
- **Q24:** How do you handle gaps in a daily series — `fillna` vs interpolation risks?

### Recap
1. What does `errors="coerce"` give you, and why is that *useful*?
2. `resample("M").sum()` vs `groupby(month).sum()` — same or different?
3. How do you compute month-over-month growth in one call?

➡️ Open **`practice.ipynb`**.